# Ramen：联合语义自适应 3DGS 对比测试

运行时请选择 **L4 GPU**。默认执行 30k 正式对比。数据、预处理、检查点和结果写入 Google Drive；断线后重新运行全部单元格会从最近检查点继续。测试集采用 LERF-Mask Ramen 官方 `test_*.jpg` 和 `test_mask`，输出 PSNR、SSIM、mIoU、Boundary-IoU、高斯总数和三级高斯数量。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 运行时 > 更改运行时类型 中选择 L4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%cd /content
![ -d /content/gaussian-splatting/.git ] && git -C /content/gaussian-splatting pull --ff-only || git clone --recursive https://github.com/Xuyw041006-arch/gaussian-splatting.git /content/gaussian-splatting
%cd /content/gaussian-splatting
!git submodule update --init --recursive
!pip -q install plyfile open-clip-torch scikit-learn ftfy regex opencv-python-headless
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install ./submodules/diff-gaussian-rasterization ./submodules/simple-knn ./submodules/fused-ssim
!python -m unittest discover -s tests -p 'test_*.py' -v
!python -m compileall -q scene semantic scripts train.py preprocess_semantics.py

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PERSIST_ROOT = Path('/content/drive/MyDrive/semantic_adaptive_3dgs/ramen_full_30k')
ASSETS = PERSIST_ROOT / 'assets'
DATA_ROOT = PERSIST_ROOT / 'data'
ASSETS.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RAMEN_ZIP = ASSETS / 'ramen.zip'
SAM_CHECKPOINT = ASSETS / 'sam_vit_h_4b8939.pth'
SCENE = DATA_ROOT / 'ramen'
!test -f {RAMEN_ZIP} || wget -q --show-progress -O {RAMEN_ZIP} https://huggingface.co/mqye/Gaussian-Grouping/resolve/main/data/lerf_mask/ramen.zip
!test -d {SCENE} || unzip -q {RAMEN_ZIP} -d {DATA_ROOT}
!test -f {SAM_CHECKPOINT} || wget -q --show-progress -O {SAM_CHECKPOINT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
print('持久化目录:', PERSIST_ROOT)

In [ ]:
MODE = 'full'
ITERATIONS = 30000 if MODE == 'full' else 1500
SEMANTIC_ITERATIONS = 5000 if MODE == 'full' else 300
SEMANTIC_START = 1000 if MODE == 'full' else 500
OUTPUT = str(PERSIST_ROOT / f'outputs_{MODE}')
print(MODE, ITERATIONS, SEMANTIC_ITERATIONS, SEMANTIC_START, OUTPUT)

In [ ]:
%cd /content/gaussian-splatting
!python scripts/run_ramen_benchmark.py --scene {SCENE} --sam_checkpoint {SAM_CHECKPOINT} --output_root {OUTPUT} --iterations {ITERATIONS} --semantic_iterations {SEMANTIC_ITERATIONS} --semantic_start {SEMANTIC_START} --resume
import json
from pathlib import Path
comparison = json.loads((Path(OUTPUT) / 'comparison.json').read_text())
comparison

In [ ]:
from IPython.display import display
from PIL import Image
predictions = sorted((Path(OUTPUT) / 'eval_joint').glob('*.png'))[:6]
for path in predictions:
    print(path.name)
    display(Image.open(path))